In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


In [2]:
!git clone -q https://github.com/facebookresearch/vggt.git /kaggle/working/vggt
!pip install -q trimesh

In [6]:
import sys, glob
sys.path.insert(0, "/kaggle/working/vggt")
import torch
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images

image_paths = sorted(glob.glob("/kaggle/input/**/*.jpg", recursive=True))
print(len(image_paths))

device = "cuda"
dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
model = VGGT.from_pretrained("facebook/VGGT-1B").to(device)
images = load_and_preprocess_images(image_paths).to(device)
print(images.shape)

23
torch.Size([23, 3, 518, 518])


In [7]:
import time
torch.cuda.synchronize()
start = time.perf_counter()
with torch.no_grad():
    with torch.autocast("cuda", dtype=dtype):
        predictions = model(images)
torch.cuda.synchronize()
print(f"Inference: {time.perf_counter() - start:.2f} s")
print(f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

Inference: 17.13 s
Peak GPU memory: 11.03 GB


In [8]:
import numpy as np
import trimesh
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
from vggt.utils.geometry import unproject_depth_map_to_point_map

extrinsic, intrinsic = pose_encoding_to_extri_intri(predictions["pose_enc"], images.shape[-2:])
extrinsic = extrinsic.squeeze(0).float().cpu().numpy()
intrinsic = intrinsic.squeeze(0).float().cpu().numpy()
depth = predictions["depth"].squeeze(0).float().cpu().numpy()
depth_conf = predictions["depth_conf"].squeeze(0).float().cpu().numpy()

points = unproject_depth_map_to_point_map(depth, extrinsic, intrinsic)
colors = (images.permute(0, 2, 3, 1).float().cpu().numpy() * 255).astype(np.uint8)

mask = depth_conf >= 5.0
pts, rgb = points[mask], colors[mask]
if len(pts) > 500_000:
    idx = np.random.default_rng(0).choice(len(pts), 500_000, replace=False)
    pts, rgb = pts[idx], rgb[idx]

trimesh.PointCloud(pts, colors=rgb).export("/kaggle/working/scene1_vggt.ply")
np.savez("/kaggle/working/scene1_vggt_cameras.npz", extrinsic=extrinsic, intrinsic=intrinsic, depth=depth, depth_conf=depth_conf, image_paths=np.array(image_paths))
print(len(pts), "points saved")

357298 points saved
